# 04 — Genetic Algorithm Ensemble Optimization
## GA Hyperparameter Tuning for XGB-RF Ensemble

This notebook uses a Genetic Algorithm (DEAP) to optimize the hyperparameters
of the XGBoost-Random Forest soft voting ensemble.

Produces: V12 (GA convergence curve), V13 (GA best params table)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

from src.models.ga_optimizer import ga_optimize

FIGURES_DIR = Path('../outputs/figures')
TABLES_DIR = Path('../outputs/tables')
MODELS_DIR = Path('../outputs/models')
PROCESSED_DIR = Path('../data/processed')

for d in [FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Load Data

In [ ]:
# Use BAF dataset for GA optimization
X_train = pd.read_csv(PROCESSED_DIR / 'baf_X_train.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'baf_y_train.csv').squeeze()

# Subsample for faster GA (full dataset is too slow for 30 gens x 20 pop)
SUBSAMPLE = 50_000
if len(X_train) > SUBSAMPLE:
    rng = np.random.RandomState(42)
    idx = rng.choice(len(X_train), SUBSAMPLE, replace=False)
    X_ga = X_train.iloc[idx].reset_index(drop=True)
    y_ga = y_train.iloc[idx].reset_index(drop=True)
    print(f'Subsampled {SUBSAMPLE:,} from {len(X_train):,} for GA')
else:
    X_ga, y_ga = X_train, y_train
    print(f'Using full training set: {len(X_train):,}')

## 2. Run Genetic Algorithm Optimization

In [ ]:
best_params, logbook = ga_optimize(
    X_ga, y_ga,
    n_generations=30,
    pop_size=20,
    cv_folds=3,
    random_state=42
)

print('\nOptimized Parameters:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

## 3. V12 — GA Convergence Curve

In [ ]:
gen = logbook.select('gen')
max_fitness = logbook.select('max')
avg_fitness = logbook.select('avg')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(gen, max_fitness, 'b-o', linewidth=2, markersize=4, label='Best (Max)')
ax.plot(gen, avg_fitness, 'r--s', linewidth=1.5, markersize=3, label='Average')
ax.set_xlabel('Generation', fontsize=12)
ax.set_ylabel('AUPRC (3-Fold CV)', fontsize=12)
ax.set_title('Genetic Algorithm Convergence', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ga_convergence.png', dpi=300)
plt.show()
print('Saved: ga_convergence.png')

## 4. V13 — Best Hyperparameters Table

In [ ]:
params_df = pd.DataFrame([
    {'Parameter': 'XGBoost n_estimators', 'Value': best_params['xgb_n_estimators']},
    {'Parameter': 'XGBoost max_depth', 'Value': best_params['xgb_max_depth']},
    {'Parameter': 'XGBoost learning_rate', 'Value': f"{best_params['xgb_learning_rate']:.4f}"},
    {'Parameter': 'XGBoost subsample', 'Value': f"{best_params['xgb_subsample']:.3f}"},
    {'Parameter': 'Random Forest n_estimators', 'Value': best_params['rf_n_estimators']},
    {'Parameter': 'Random Forest max_depth', 'Value': best_params['rf_max_depth']},
])
params_df.to_csv(TABLES_DIR / 'ga_best_params.csv', index=False)
print(params_df.to_string(index=False))
print('\nSaved: ga_best_params.csv')

## 5. Train Optimized Ensemble and Save

In [ ]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Train on full training data with optimized params
xgb_opt = XGBClassifier(
    n_estimators=best_params['xgb_n_estimators'],
    max_depth=best_params['xgb_max_depth'],
    learning_rate=best_params['xgb_learning_rate'],
    subsample=best_params['xgb_subsample'],
    eval_metric='aucpr', random_state=42,
    use_label_encoder=False, n_jobs=-1,
)
rf_opt = RandomForestClassifier(
    n_estimators=best_params['rf_n_estimators'],
    max_depth=best_params['rf_max_depth'],
    random_state=42, n_jobs=-1,
)
ensemble_opt = VotingClassifier(
    estimators=[('xgb', xgb_opt), ('rf', rf_opt)],
    voting='soft', n_jobs=-1,
)

print('Training GA-optimized ensemble on full training data...')
ensemble_opt.fit(X_train, y_train)

# CV score on full data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(ensemble_opt, X_train, y_train, cv=cv,
                            scoring='average_precision', n_jobs=-1)
print(f'GA-Optimized Ensemble CV AUPRC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

# Save
joblib.dump(ensemble_opt, MODELS_DIR / 'ga_optimized_ensemble.joblib')
joblib.dump(best_params, MODELS_DIR / 'ga_best_params.joblib')
print('Saved: ga_optimized_ensemble.joblib, ga_best_params.joblib')